# BioClinical encoder + Laya distill, then medical RLCD smoke

1. Swap ModernBERT for BioClinical ModernBERT-large.
2. Distill frozen Laya soft targets onto the head (encoder frozen) using typed-decisions.
3. Pure-RLCD medical smoke (4k MedMCQA + 1k MedNLI, 1 epoch).

**Settings:** GPU T4 x2, Internet on.

In [ ]:
import os, sys, subprocess, shutil
from pathlib import Path
print('python', sys.version)
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), 'n_gpu', torch.cuda.device_count())
if torch.cuda.device_count() < 1:
    raise SystemExit('No GPU. Set Accelerator to GPU T4 x2.')
os.environ['USE_TF'] = '0'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

In [ ]:
%pip -q install -U 'laya>=0.3.7' 'transformers>=4.48.0' 'datasets>=3.0.0' safetensors huggingface_hub accelerate scipy pyarrow pandas tabulate PyYAML

In [ ]:
REPO = 'https://github.com/Priyanshu-5257/laya-medical-finetune.git'
REPO_DIR = Path('/kaggle/working/repo')
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.check_call(['git', 'clone', '--branch', 'main', '--depth', '1', REPO, str(REPO_DIR)])
subprocess.check_call(['git', '-C', str(REPO_DIR), 'log', '-1', '--oneline'])

In [ ]:
def run(cmd, cwd=None, env=None):
    print('+', *cmd, flush=True)
    e = os.environ.copy()
    if env: e.update(env)
    p = subprocess.Popen(cmd, cwd=cwd, env=e, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end='', flush=True)
    rc = p.wait()
    if rc != 0:
        raise SystemExit(rc)

run(['bash', 'scripts/run_bioclinical_try.sh'], cwd=str(REPO_DIR), env={'WORK': '/kaggle/working'})

In [ ]:
from pathlib import Path
for name in ('summary_distill.json', 'summary_bioclinical_medical.json'):
    p = Path('/kaggle/working') / name
    print('====', name)
    print(p.read_text() if p.exists() else 'MISSING')